# MOCK INTERVIEWER — DAY 1

In [ ]:
# Goal: Parse a resume PDF and generate 5 interview
# questions based on its content + job role
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from groq import Groq

load_dotenv()
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


# PART 1: Parse the Resume


In [ ]:
print("=" * 55)
print("PART 1: Resume Parser")
print("=" * 55)

def parse_resume(pdf_path):
    loader=PyPDFLoader(pdf_path)
    pages=loader.load()
    full_text = "\n".join(page.page_content for page in pages)
    # Basic cleanup
    full_text = "\n".join(
        line.strip() for line in full_text.splitlines()
        if line.strip()  # remove empty lines
    )
    return full_text

RESUME_PATH = "Resume_Garv.pdf"

resume_text=parse_resume(RESUME_PATH)
print(f"\nResume parsed successfully!")
print(f"Total characters: {len(resume_text)}")
print(f"\nPreview:\n{resume_text[:500]}...")



PART 1: Resume Parser

Resume parsed successfully!
Total characters: 3427

Preview:
Garv Rawat
+91 8527683067|rawatgarv46@gmail.com |LinkedIn |GitHub
Summary
Aspiring AI/ML engineer with hands-on experience building and deploying machine learning models, NLP pipelines, and
LLM-powered applications. Proficient in Python, Scikit-learn, LangChain, and Flask for end-to-end model development
and API deployment. Experienced in full-stack development with React.js and Node.js, enabling seamless integration of AI
models into production web applications.
Education
V ellore Institute of ...


# Part 2: Question Generator

In [6]:
print("\n" + "=" * 55)
print("PART 2: Question Generator")
print("=" * 55)

#available job roles
ROLES = {
    "aiml":      "AI/ML Engineer",
    "webdev":    "Full Stack Web Developer",
    "fullstack": "Full Stack Developer",
    "backend":   "Backend Developer",
    "data":      "Data Scientist"
}

def generate_questions(resume_text,role_key,n_questions=5):
    """
    Generate interview questions based on resume content + role.
    Returns list of question strings.
    """
    role_name=ROLES.get(role_key,role_key)
    prompt=f"""You are an experienced technical interviewer hiring for a {role_name} position.

Based on the candidate's resume below, generate exactly {n_questions} interview questions.

Rules:
- Mix of technical and project-based questions
- Questions must be SPECIFIC to what's on their resume
  (mention their actual projects, skills, and technologies)
- Start with easier questions, end with harder ones
- Each question on a new line starting with a number and period
- No explanations, just the questions

Resume:
{resume_text}

Generate {n_questions} interview questions:"""
    
    response=groq_client.chat.completions.create(
        model      = "llama-3.3-70b-versatile",
        messages   = [{"role": "user", "content": prompt}],
        max_tokens = 500,
        temperature= 0.7   # slightly creative for varied questions
    )
    raw=response.choices[0].message.content.strip()

     # Parse questions into a list
    questions=[]
    for line in raw.splitlines():
        line = line.strip()
        # Keep lines that start with a nwumber
        if line and line[0].isdigit():#meaning its a question
            #Remove "1. "prefix
            question = line.split(".",1)[-1].strip()
            if question:
                questions.append(question)
    return questions[:n_questions]

# Test question generation for different roles
print("\nGenerating questions for AI/ML role...")
questions_aiml = generate_questions(resume_text, "aiml")
print(f"\n{'='*50}")
print("AI/ML Interview Questions:")
print(f"{'='*50}")
for i, q in enumerate(questions_aiml, 1):
    print(f"\nQ{i}: {q}")


print("\n" + "-"*50)
print("\nGenerating questions for Web Dev role...")
questions_web = generate_questions(resume_text, "webdev")

print(f"\n{'='*50}")
print("Web Dev Interview Questions:")
print(f"{'='*50}")
for i, q in enumerate(questions_web, 1):
    print(f"\nQ{i}: {q}")



PART 2: Question Generator

Generating questions for AI/ML role...

AI/ML Interview Questions:

Q1: Can you explain how you used Scikit-learn and TF-IDF to train the sentiment classifier in the SocialSphere project, and what kind of results did you achieve in terms of accuracy and F1 score?

Q2: How did you implement the conversation memory with a 10-message sliding window in the DocChat project, and what were some of the challenges you faced while integrating it with the React.js frontend?

Q3: You mentioned using LangChain and ChromaDB to build a multi-document RAG pipeline in DocChat - can you walk me through the process of how you retrieved the top-3 most relevant chunks per query using cosine similarity over sentence-transformer embeddings?

Q4: In the SocialSphere project, you deployed a Flask REST API to automatically label social media posts as positive or negative in real time - how did you handle issues like data imbalance, overfitting, or class imbalance in your sentiment c

# PART 3: Session Manager


In [11]:
# Keeps track of the interview state

print("\n" + "=" * 55)
print("PART 3: Session Manager")
print("=" * 55)

# 🔑 A session stores everything about one interview
# We'll use this in Flask 

def create_session(resume_text,role_key):
    """Create a new interview session"""
    questions=generate_questions(resume_text,role_key)
    return {
        "role":ROLES.get(role_key,role_key),
        "resume_text":resume_text,
        "questions":questions,
        "answers":[],# filled as user answers
        "evaluations":[], # filled as AI evaluates
        "current_q":0,# which question we're on
        "done":False
    }

def get_current_question(session):
    """Get the current question"""
    idx=session["current_q"]
    if idx>=len(session["questions"]):
        return None
    return {
        "question_number":idx + 1,
        "total":len(session["questions"]),
        "question":session["questions"][idx]
    }

def submit_answers(session,answer):
    """Add answer to session, move to next question"""
    session["answers"].append(answer)
    session["current_q"]+=1
    if session["current_q"]>=len(session["questions"]):
        session["done"]=True
    return session


print("\nCreating interview session...")
session = create_session(resume_text, "aiml")

print(f"Role: {session['role']}")
print(f"Questions generated: {len(session['questions'])}")
print(f"\nFirst question: {get_current_question(session)['question']}")

# Simulate answering
session = submit_answers(session ,"I would use dropout and cross validation")
print(f"\nAfter answering Q1:")
print(f"Current question index: {session['current_q']}")
print(f"Next question: {get_current_question(session)['question']}")
print(f"Done: {session['done']}")





PART 3: Session Manager

Creating interview session...
Role: AI/ML Engineer
Questions generated: 5

First question: Can you explain how you used Scikit-learn and TF-IDF to train the sentiment classifier in your SocialSphere project, and what kind of results did you achieve in terms of accuracy and F1 score?

After answering Q1:
Current question index: 1
Next question: How did you implement the conversation memory feature in your DocChat project, and what was the reasoning behind choosing a 10-message sliding window for the chat history?
Done: False


# PART 4: Save parsed resume


In [12]:
# So Flask doesn't need to re-parse on every request
print("\n" + "=" * 55)
print("PART 4: Save Resume Text")
print("=" * 55)

import json

def save_resume(resume_text,filepath="parsed_resume.json"):
    with open(filepath,"w") as f:
        json.dump({"text":resume_text},f)
    print(f"✅ Saved to {filepath}")

def load_resume(filepath="parsed_resume.json"):
    with open(filepath) as f:
        return json.load(f)["text"]

save_resume(resume_text)
loaded=load_resume()
print(f"Loaded back: {len(loaded)} charaters ✅")




PART 4: Save Resume Text
✅ Saved to parsed_resume.json
Loaded back: 3427 charaters ✅


# DAY 1 CHALLENGE


In [13]:
print("\n" + "=" * 55)
print("DAY 1 CHALLENGE")
print("=" * 55)

print("""
1. Use YOUR actual resume PDF (not the sample).
   Set RESUME_PATH to your real resume.
   Are the questions specific to your actual projects?

   # Answer:
   # If the resume contains detailed projects, skills,
   # technologies, and achievements, the AI generates
   # more personalized and project-specific interview questions.
   # Example:
   # If resume mentions "RAG Chatbot using LangChain",
   # AI may ask:
   # "Explain how vector databases work in your RAG pipeline."



2. Generate questions for 3 different roles.
   Which role gives the most relevant questions?
   Which questions would you struggle to answer?

   # Answer:
   # Different roles focus on different skills.
   #
   # Example:
   # - AI/ML Role → ML algorithms, deep learning, NLP
   # - Backend Role → APIs, databases, Flask, scaling
   # - Frontend Role → React, UI, state management
   #
   # Most relevant role:
   # Usually the one matching your strongest projects.
   #
   # Questions you struggle with:
   # These reveal weak areas that need preparation.



3. Try passing a very short resume (just name + skills).
   Does the question quality drop?
   What does this tell you about resume quality and AI?

   # Answer:
   # Yes, question quality usually drops.
   #
   # AI depends heavily on resume context.
   # Less information → more generic questions.
   #
   # This shows:
   # Better resumes help AI generate better,
   # more targeted interview preparation.



4. Most important:
   Look at the generated questions for your real resume.
   For each one — can you answer it?
   The ones you can't answer → add to my_notes.md
   and prepare those answers before your interview.

   # Answer:
   # This helps identify knowledge gaps.
   #
   # If you cannot confidently answer a question,
   # it means:
   # - You may not fully understand your project
   # - Or need revision in that topic
   #
   # Writing them into notes creates a focused
   # interview preparation checklist.
""")


DAY 1 CHALLENGE

1. Use YOUR actual resume PDF (not the sample).
   Set RESUME_PATH to your real resume.
   Are the questions specific to your actual projects?

   # Answer:
   # If the resume contains detailed projects, skills,
   # technologies, and achievements, the AI generates
   # more personalized and project-specific interview questions.
   # Example:
   # If resume mentions "RAG Chatbot using LangChain",
   # AI may ask:
   # "Explain how vector databases work in your RAG pipeline."



2. Generate questions for 3 different roles.
   Which role gives the most relevant questions?
   Which questions would you struggle to answer?

   # Answer:
   # Different roles focus on different skills.
   #
   # Example:
   # - AI/ML Role → ML algorithms, deep learning, NLP
   # - Backend Role → APIs, databases, Flask, scaling
   # - Frontend Role → React, UI, state management
   #
   # Most relevant role:
   # Usually the one matching your strongest projects.
   #
   # Questions you struggl

# NOtes

In [ ]:
# ============================================================
# MOCK INTERVIEWER — WORKING SUMMARY / NOTES
# ============================================================

# Overall Goal:
# This project creates an AI-powered mock interviewer.
#
# Workflow:
# Resume PDF
#      ↓
# Extract text from PDF
#      ↓
# Send resume text + job role to LLM
#      ↓
# LLM generates interview questions
#      ↓
# Session tracks answers and progress
#      ↓
# Resume text saved for future use



# ============================================================
# IMPORTS
# ============================================================

# os
# → Access environment variables

# load_dotenv()
# → Loads GROQ_API_KEY from .env file

# PyPDFLoader
# → Reads PDF pages and extracts text

# RecursiveCharacterTextSplitter
# → Splits large text into chunks
# (imported for future scaling, not heavily used here)

# Groq
# → Connects Python code to Groq LLM API



# ============================================================
# GROQ CLIENT SETUP
# ============================================================

# Creates connection to Groq servers
# using API key from .env

# groq_client.chat.completions.create(...)
# is the actual LLM call



# ============================================================
# PART 1 — RESUME PARSER
# ============================================================

# Goal:
# Convert resume PDF into clean text.

# Workflow:
# PDF
#   ↓
# PyPDFLoader loads pages
#   ↓
# Extract page text
#   ↓
# Combine pages into one large string
#   ↓
# Remove empty lines
#   ↓
# Return clean resume text


# parse_resume(pdf_path)

# pages = loader.load()
# → Reads all PDF pages

# "\n".join(...)
# → Combines all pages together

# line.strip()
# → Removes extra spaces

# if line.strip()
# → Removes blank lines


# Result:
# AI receives resume as plain text
# instead of PDF file.



# ============================================================
# SAMPLE RESUME CREATION
# ============================================================

# If resume.pdf does not exist,
# code creates a temporary sample resume.

# FPDF library:
# → Used to generate PDFs using Python

# multi_cell(...)
# → Writes multiple lines into PDF


# Important:
# In real projects,
# user uploads their actual resume PDF.



# ============================================================
# PART 2 — QUESTION GENERATOR
# ============================================================

# Goal:
# Generate interview questions
# based on:
# 1. Resume content
# 2. Job role


# ROLES dictionary:
# Maps short role keys to full role names

# Example:
# "aiml" → "AI/ML Engineer"


# generate_questions(...)

# Step 1:
# Get role name

# Step 2:
# Build detailed prompt

# Prompt contains:
# - Job role
# - Resume text
# - Instructions for AI

# Example instructions:
# - Ask technical questions
# - Mention actual projects
# - Start easy → end hard


# ============================================================
# HOW THE LLM WORKS HERE
# ============================================================

# resume_text + prompt
#          ↓
# Sent to Groq API
#          ↓
# Llama model reads resume
#          ↓
# Generates interview questions
#          ↓
# Python receives response


# ============================================================
# GROQ API CALL
# ============================================================

# groq_client.chat.completions.create(...)

# model
# → Which LLM to use

# messages
# → Prompt sent to model

# max_tokens
# → Maximum response size

# temperature
# → Creativity level
#
# 0.1 → More focused
# 0.7 → More creative


# ============================================================
# QUESTION PARSING
# ============================================================

# LLM returns one large text response.

# Example:
# 1. Explain TF-IDF
# 2. How did you use Flask?
# ...

# Code splits lines
# and extracts only numbered questions.


# line[0].isdigit()
# → Keeps only numbered lines

# split(".", 1)
# → Removes "1. " from question


# Final Output:
# Python list of questions


# ============================================================
# enumerate()
# ============================================================

# Used while printing questions

# Example:
# for i, q in enumerate(questions, 1):

# Gives:
# i → question number
# q → actual question

# Output:
# Q1: ...
# Q2: ...


# ============================================================
# PART 3 — SESSION MANAGER
# ============================================================

# Goal:
# Keep track of interview progress.

# Session stores:
# - Role
# - Resume text
# - Questions
# - User answers
# - AI evaluations
# - Current question number
# - Interview completion status


# create_session(...)

# Creates interview state object

# Similar to:
# User login session
# but for interview tracking.


# ============================================================
# SESSION STRUCTURE
# ============================================================

# {
#   "role": ...
#   "questions": [...]
#   "answers": [...]
#   "current_q": 0
#   "done": False
# }


# ============================================================
# get_current_question()
# ============================================================

# Returns current active question.

# session["current_q"]
# stores question index.

# Example:
# current_q = 0
# → First question


# ============================================================
# submit_answer()
# ============================================================

# Adds user answer into session.

# Then:
# current_q += 1

# Moves interview to next question.

# If all questions answered:
# done = True


# ============================================================
# PART 4 — SAVE RESUME TEXT
# ============================================================

# Goal:
# Avoid parsing PDF repeatedly.

# Instead of:
# PDF → parse every request

# We do:
# Parse once
#   ↓
# Save extracted text into JSON


# save_resume(...)

# json.dump(...)
# → Saves resume text into file


# load_resume(...)

# json.load(...)
# → Reads saved resume text


# Benefit:
# Faster backend requests
# Better performance



# ============================================================
# COMPLETE PROJECT FLOW
# ============================================================

# User Resume PDF
#        ↓
# parse_resume()
#        ↓
# Extract clean text
#        ↓
# generate_questions()
#        ↓
# Prompt + resume sent to LLM
#        ↓
# LLM generates questions
#        ↓
# create_session()
#        ↓
# Track interview progress
#        ↓
# Save resume text for reuse



# ============================================================
# MAIN AI CONCEPTS USED
# ============================================================

# 1. Prompt Engineering
# Carefully designing prompts for better AI output

# 2. LLM API Integration
# Python communicating with Groq-hosted Llama model

# 3. PDF Parsing
# Extracting structured text from resumes

# 4. Session Management
# Tracking user interview progress

# 5. Serialization
# Saving parsed data into JSON files



# ============================================================
# INTERVIEW UNDERSTANDING
# ============================================================

# This project demonstrates:
#
# - API integration
# - LLM usage
# - Prompt engineering
# - Python backend logic
# - PDF processing
# - State/session management
#
# Good project for:
# - AI/ML roles
# - Backend roles
# - Full stack AI projects